# Concept Probe — 100 concepts: richer independent factual descriptions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1111_qwen_probe_100concepts_v2/qwen_probe_100concepts_v2.ipynb)

**Goal:** Build a better-powered exploratory benchmark from ~100 concepts using **8 training and 8 evaluation factual sentences per concept**, six Y comparisons spanning within-category near/mid/far semantic distance, three layers, and randomized negative controls. We do **not** increase sample size by duplicating paraphrased wrappers.

- Sources: English Wikipedia for training; Simple English Wikipedia preferred for evaluation, with marked English Wikipedia held-out fallback. Sentence-level sources and revisions are retained. Source-held-out claims apply only to Simple Wikipedia subset.
- Concepts lacking sufficient unique sentences are **excluded**, never padded with copies. Review the `data_coverage.csv` report before accepting the run.
- **Model:** `Qwen/Qwen2.5-0.5B-Instruct` pinned revision, layers 22/23/24, `assistant_start` token. **Distance:** `all-mpnet-base-v2`, based only on title + training descriptions.
- **Important:** Wikipedia text can describe related entities, include spurious clues, and contain errors. Automatic filtering must be supplemented by manually reviewing the shortlisted concepts.
- **Interpretation:** Probe AUC measures *relative X-vs-Y decodability*, not absolute concept activation or the effectiveness of suppression.

Execution requires a **T4 GPU** and Colab secret `GITHUB_RESULTS_TOKEN` with repository Contents read/write. The notebook commits immutable outputs into its own directory. If input sources or configuration change, set a **new `RUN_ID`**.


In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" "sentence-transformers>=3,<6" pandas numpy requests tqdm matplotlib tabulate


In [ ]:
import os, io, json, re, math, time, hashlib, zipfile, random, base64, itertools, unicodedata
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import numpy as np, pandas as pd, requests, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from google.colab import userdata

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
REPO='ryouy/GraduationThesis'
RUN_ID='20261111_qwen100_richer_v1'
ROOT='1111_qwen_probe_100concepts_v2'
DEST=f'{ROOT}/results/{RUN_ID}'
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
REVISION='7ae557604adf67be50417f59c2c2f167def9a775'
EMBEDDER='sentence-transformers/all-mpnet-base-v2'
MIN_PER_SPLIT=8
TRAIN_PER_CONCEPT=8
EVAL_PER_CONCEPT=8
LAYERS=[22,23,24]
BATCH_SIZE=64
BATCHES_PER_CHECKPOINT=4
WORK=Path('/content/qwen_probe_100_richer')/RUN_ID
CACHE=WORK/'cache';CACHE.mkdir(parents=True,exist_ok=True)
OUT=WORK/'results';OUT.mkdir(parents=True,exist_ok=True)
try:TOKEN=userdata.get('GITHUB_RESULTS_TOKEN')
except Exception:TOKEN=None
assert TOKEN,'Set GITHUB_RESULTS_TOKEN in Colab Secrets; grant notebook access.'
SESSION=requests.Session()
SESSION.headers.update({'Authorization':'Bearer '+TOKEN,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
API=f'https://api.github.com/repos/{REPO}/contents/'

def remote_meta(path):
    res=SESSION.get(API+path,timeout=90)
    if res.status_code==404:return None
    res.raise_for_status();return res.json()

def github_get(path):
    meta=remote_meta(path)
    if meta is None:return None
    response=SESSION.get(meta['download_url'],timeout=180)
    response.raise_for_status();return response.content

def github_save(path,blob,description):
    sha=hashlib.sha1(b'blob '+str(len(blob)).encode()+b'\0'+blob).hexdigest()
    old=remote_meta(path)
    if old is not None:
        if old['sha']!=sha:raise RuntimeError('Remote immutable file differs; choose a NEW RUN_ID: '+path)
        print('Already verified:',path);return
    payload={'branch':'main','message':description,'content':base64.b64encode(blob).decode('ascii')}
    response=SESSION.put(API+path,json=payload,timeout=180)
    response.raise_for_status();print('GitHub saved:',path)

# Sources are downloaded before GPU work, then frozen by full-source hash.
SOURCE_SESSION=requests.Session()
SOURCE_SESSION.headers.update({'User-Agent':'AizuUniversityConceptProbeResearch/1.0 (academic reproducibility; contact via github.com/ryouy/GraduationThesis)'})


## 1. Concept inventory and factual stimulus sourcing

The inventory of 100 concepts is copied verbatim from the prior experiment, but previous training/evaluation sentences are **not** used as evaluation samples. Each concept needs 8 different sentences from English Wikipedia for training and 8 held-out sentences for evaluation. Simple English Wikipedia is prioritized; when it has insufficient text, later non-overlapping English Wikipedia sentences are used with explicit fallback provenance. All links, revisions and sentence texts are saved for audit. Some concepts may fail this stringent requirement; the actual sample size is reported rather than silently replaced.


In [ ]:
BANK = {'elephant': {'group': 'land_mammals', 'facts': ['A massive herbivore carries a flexible trunk.', 'Large ears release heat in warm regions.', 'Families travel together under older females.', 'Thick pillars support an exceptionally heavy body.', 'Ivory tusks can grow from upper teeth.']}, 'giraffe': {'group': 'land_mammals', 'facts': ['An extremely tall browser reaches high foliage.', 'A spotted coat breaks up its silhouette.', 'Long legs make drinking from pools awkward.', 'Individuals feed from branches far above ground.', 'Newborns stand soon after a considerable fall.']}, 'lion': {'group': 'land_mammals', 'facts': ['A large social predator lives in family prides.', 'Adult males often grow a heavy mane.', 'Groups cooperate when pursuing large prey.', 'Territorial roaring travels over long distances.', 'Resting occupies much of the hot day.']}, 'tiger': {'group': 'land_mammals', 'facts': ['A solitary striped predator hunts by ambush.', 'Orange fur carries bold dark bands.', 'Swimming is common among these powerful cats.', 'Individuals occupy territories in dense habitats.', 'A silent approach precedes a sudden attack.']}, 'wolf': {'group': 'land_mammals', 'facts': ['A pack hunter communicates with howls.', 'Long-distance travel helps track hoofed prey.', 'Social groups often center on breeding adults.', 'Scent marks help establish territorial boundaries.', 'Thick seasonal fur protects against winter cold.']}, 'fox': {'group': 'land_mammals', 'facts': ['A small wild canine often hunts rodents alone.', 'A bushy tail assists with balance and warmth.', 'Some species adapt well to cities.', 'Sharp hearing locates prey below snow or grass.', 'The animal frequently rests in an underground den.']}, 'rabbit': {'group': 'land_mammals', 'facts': ['A long-eared herbivore escapes through fast hops.', 'Powerful rear legs propel rapid bounds.', 'Many species rely on burrows for shelter.', 'Constantly growing incisors grind plant material.', 'Eyes on the sides of the head detect threats.']}, 'deer': {'group': 'land_mammals', 'facts': ['A hoofed browser or grazer often has antlers.', 'Many males shed and regrow bony headgear annually.', 'Herds move between feeding and sheltering sites.', 'Alert ears and rapid sprints aid predator avoidance.', 'Young animals may hide in vegetation at first.']}, 'bear': {'group': 'land_mammals', 'facts': ['A heavy omnivore uses strong paws to forage.', 'Some populations spend winter in prolonged dormancy.', 'Thick fur and stored fat aid cold-weather survival.', 'Individuals may catch fish in shallow streams.', 'Cubs accompany their mothers while learning food sources.']}, 'horse': {'group': 'land_mammals', 'facts': ['A large hoofed grazer is domesticated for riding.', 'Long legs support sustained movement over grassland.', 'Herd members communicate through posture and calls.', 'A mane runs along the upper neck.', 'Hooves require regular care in human-managed settings.']}, 'eagle': {'group': 'birds', 'facts': ['A powerful raptor soars on rising air.', 'Hooked beaks tear food into pieces.', 'Large talons grip captured animals.', 'Keen vision helps locate prey from high above.', 'Many pairs maintain extensive nesting territories.']}, 'penguin': {'group': 'birds', 'facts': ['A flightless seabird uses flippers to swim.', 'Dense feathers insulate against chilly seawater.', 'Colonies gather on rocky or icy coasts.', 'Upright walking contrasts with agile underwater hunting.', 'Adults may take turns guarding an egg.']}, 'owl': {'group': 'birds', 'facts': ['A nocturnal hunter often flies almost silently.', 'Forward-facing eyes aid depth judgment.', 'Soft-edged flight feathers reduce noise.', 'A flexible neck allows a wide viewing angle.', 'Regurgitated pellets contain undigested bones.']}, 'parrot': {'group': 'birds', 'facts': ['A colorful bird has a curved strong beak.', 'Some species imitate human speech sounds.', 'Grasping feet manipulate seeds and fruit.', 'Social flocks communicate with loud calls.', 'Long-lived individuals can learn complicated routines.']}, 'swan': {'group': 'birds', 'facts': ['A large waterbird has an elegant curved neck.', 'Broad webbed feet propel it across lakes.', 'Pairs often defend nesting areas vigorously.', 'Long necks reach submerged aquatic plants.', 'Young birds travel close to protective adults.']}, 'flamingo': {'group': 'birds', 'facts': ['A wading bird uses a specialized filter-feeding bill.', 'Pink hues come partly from pigments in food.', 'Large groups gather in shallow salty water.', 'One-legged resting is frequently observed.', 'Head-down feeding strains tiny organisms from water.']}, 'peacock': {'group': 'birds', 'facts': ['A male gamebird spreads a dramatic tail display.', 'Iridescent eyespots decorate long ornamental plumes.', 'Loud calls carry through woodland edges.', 'Courtship involves fanning and vibrating feathers.', 'The female has comparatively subdued coloration.']}, 'sparrow': {'group': 'birds', 'facts': ['A small songbird thrives near human settlements.', 'Short conical bills crack seeds efficiently.', 'Flocks gather around bushes and feeders.', 'Quick hopping movements are common on the ground.', 'Simple cup-shaped nests fit into sheltered spaces.']}, 'woodpecker': {'group': 'birds', 'facts': ['A tree-climbing bird drums against wood.', 'A rigid tail braces the body on trunks.', 'Chisel-like bills excavate cavities.', 'Long tongues reach insects within bark.', 'Rhythmic tapping can also signal territorial claims.']}, 'albatross': {'group': 'birds', 'facts': ['A seabird glides over oceans on long wings.', 'Dynamic soaring reduces the need for flapping.', 'Individuals travel immense distances between landfalls.', 'Breeding pairs nest on remote oceanic islands.', 'A single chick may require prolonged parental care.']}, 'dolphin': {'group': 'marine_animals', 'facts': ['An intelligent marine mammal breathes at the surface.', 'Echolocation helps detect underwater objects.', 'Coordinated groups chase schools of fish.', 'A streamlined body supports rapid swimming.', 'Whistles and clicks carry social information.']}, 'shark': {'group': 'marine_animals', 'facts': ['A cartilaginous fish senses electrical fields.', 'Multiple rows of teeth can replace lost ones.', 'Some species must keep moving for water flow.', 'A dorsal fin often cuts above the surface.', 'Hunting relies on smell and specialized sensory organs.']}, 'whale': {'group': 'marine_animals', 'facts': ['A gigantic aquatic mammal surfaces to breathe.', 'Some species sieve krill through baleen plates.', 'Long migrations link feeding and breeding grounds.', 'Low-frequency calls can travel far underwater.', 'Thick blubber stores energy and provides insulation.']}, 'octopus': {'group': 'marine_animals', 'facts': ['A soft-bodied hunter has eight flexible arms.', 'Color-changing skin helps with camouflage.', 'Suckers manipulate shells and other objects.', 'The animal can squeeze through tiny openings.', 'Ink clouds may distract approaching predators.']}, 'jellyfish': {'group': 'marine_animals', 'facts': ['A gelatinous drifter propels itself by pulsing.', 'Trailing tentacles often bear stinging cells.', 'Ocean currents move these translucent animals.', 'A bell-shaped body lacks a conventional skeleton.', 'Some species glow in darkness.']}, 'seahorse': {'group': 'marine_animals', 'facts': ['A small fish swims upright in sea grass.', 'A curled tail grips stems underwater.', 'Males carry developing young in a brood pouch.', 'Tube-shaped mouths suck up tiny crustaceans.', 'Armor-like plates cover its slender body.']}, 'seal': {'group': 'marine_animals', 'facts': ['A marine mammal rests on rocks and ice.', 'Streamlined bodies dive for fish.', 'Whiskers detect water movements from nearby prey.', 'Thick fat helps conserve heat in cold seas.', 'Young pups nurse on beaches or frozen platforms.']}, 'crab': {'group': 'marine_animals', 'facts': ['A hard-shelled crustacean often walks sideways.', 'Two claws grip food and defend territory.', 'Molting allows the body to grow.', 'Gills enable breathing in aquatic environments.', 'Some kinds dig burrows in tidal sand.']}, 'starfish': {'group': 'marine_animals', 'facts': ['A seafloor animal has radiating arms.', 'Tube feet grip rocks and move the body.', 'Certain species regenerate damaged limbs.', 'Some feed by extending their stomach outside.', 'Rough spines cover the upper surface.']}, 'lobster': {'group': 'marine_animals', 'facts': ['A large crustacean shelters in rocky crevices.', 'Powerful claws can crush shells.', 'Antennae sense the nearby environment.', 'A hard exoskeleton must be shed as it grows.', 'The tail curls sharply to propel escape.']}, 'butterfly': {'group': 'insects_and_small_animals', 'facts': ['A winged insect begins life as a caterpillar.', 'Colored scales cover broad wings.', 'A coiled proboscis drinks flower nectar.', 'The pupal stage transforms the body dramatically.', 'Many species migrate or overwinter seasonally.']}, 'moth': {'group': 'insects_and_small_animals', 'facts': ['A winged insect is often active at night.', 'Feathery antennae detect faint chemical signals.', 'Many species are attracted to artificial lamps.', 'Larvae may feed on leaves or textiles.', 'Camouflaged wings blend into bark or foliage.']}, 'bee': {'group': 'insects_and_small_animals', 'facts': ['A flying pollinator gathers nectar and pollen.', 'Workers communicate food locations by movement.', 'Some colonies build wax combs for their young.', 'Hairy bodies carry pollen between blossoms.', 'A sting can defend the nest against threats.']}, 'ant': {'group': 'insects_and_small_animals', 'facts': ['A social insect lives in organized colonies.', 'Workers follow chemical trails to food.', 'Underground chambers shelter developing larvae.', 'Specialized individuals divide foraging and defense.', 'Many species carry loads heavier than themselves.']}, 'spider': {'group': 'insects_and_small_animals', 'facts': ['An eight-legged predator produces silk.', 'Webs can trap flying insects.', 'Venom subdues small prey.', 'Some species actively chase instead of waiting.', 'External digestion helps consume captured food.']}, 'scorpion': {'group': 'insects_and_small_animals', 'facts': ['A desert arthropod has a curved stinging tail.', 'Large pincers grasp prey.', 'Fluorescence appears under ultraviolet light.', 'Many individuals hide beneath rocks during daytime.', 'Sensory hairs detect vibrations in the ground.']}, 'beetle': {'group': 'insects_and_small_animals', 'facts': ['A six-legged insect has hardened wing covers.', 'Protective cases shelter delicate flying wings.', 'Many species recycle decaying wood or dung.', 'Larvae often look unlike the adult form.', 'Some show spectacular horns or metallic colors.']}, 'dragonfly': {'group': 'insects_and_small_animals', 'facts': ['A fast flying insect hunts above ponds.', 'Two pairs of long wings move independently.', 'Large compound eyes detect moving prey.', 'Aquatic juveniles develop beneath the water.', 'Adults can hover and change direction abruptly.']}, 'mosquito': {'group': 'insects_and_small_animals', 'facts': ['A slender insect develops in standing water.', 'Some females seek blood for egg production.', 'A high-pitched buzz accompanies flight.', 'Larvae wriggle near the water surface.', 'Carbon dioxide and body heat guide host seeking.']}, 'cricket': {'group': 'insects_and_small_animals', 'facts': ['A jumping insect produces rhythmic chirps.', 'Hind legs propel long leaps.', 'Sound often comes from rubbing wing structures.', 'Individuals hide among grasses and stones.', 'Calls help attract mates after dark.']}, 'banana': {'group': 'fruits', 'facts': ['An elongated yellow fruit grows in hanging clusters.', 'The peel opens without cutting.', 'Commercial plants often reproduce without seeds.', 'Soft flesh becomes sweeter as it ripens.', 'Warm climates support the large leafy crop.']}, 'apple': {'group': 'fruits', 'facts': ['A crisp orchard fruit has a central seed core.', 'Trees bloom before producing their harvest.', 'Many cultivars develop red or green skins.', 'Firm flesh can be baked into pies.', 'Harvested fruit keeps well under refrigeration.']}, 'orange': {'group': 'fruits', 'facts': ['A round citrus fruit has segmented juicy flesh.', 'Aromatic oils occur in the rind.', 'Vitamin-rich juice is widely consumed at breakfast.', 'White pith separates peel from edible sections.', 'Evergreen groves thrive in warm subtropical climates.']}, 'lemon': {'group': 'fruits', 'facts': ['A sharply sour citrus fruit is often bright yellow.', 'Zest adds fragrance to cooked dishes.', 'Acidic juice helps prevent sliced produce browning.', 'Many fruits have pointed ends.', 'A squeeze provides tartness for drinks and sauces.']}, 'lime': {'group': 'fruits', 'facts': ['A small green citrus fruit has acidic juice.', 'The rind provides intense aromatic zest.', 'Wedges accompany many tropical beverages.', 'The fruit is often harvested before turning yellow.', 'Tart juice pairs with savory marinades.']}, 'grape': {'group': 'fruits', 'facts': ['Small round berries grow in bunches on vines.', 'Some cultivars are used for winemaking.', 'Thin skins surround juicy seed-bearing flesh.', 'Climbing plants are trained on trellises.', 'Dried berries become raisins.']}, 'peach': {'group': 'fruits', 'facts': ['A soft stone fruit has fuzzy skin.', 'A large hard pit protects the seed.', 'Fragrant flesh ripens during summer.', 'Blossoms appear on orchard branches in spring.', 'Tender slices bruise when handled roughly.']}, 'pear': {'group': 'fruits', 'facts': ['A tree fruit often has a narrow neck and broad base.', 'The flesh may have tiny gritty cells.', 'Many varieties soften after picking.', 'Poaching develops a delicate fragrance.', 'Orchard trees bear white spring blossoms.']}, 'pineapple': {'group': 'fruits', 'facts': ['A tropical fruit has rough patterned outer skin.', 'Spiky leaves form a crown at the top.', 'Sweet acidic flesh grows around a fibrous core.', 'A single fruit forms from fused flowers.', 'Fresh juice can tenderize some proteins.']}, 'strawberry': {'group': 'fruits', 'facts': ['A small red berry carries visible seeds outside.', 'Low plants send runners across beds.', 'White flowers precede soft fragrant fruit.', 'The harvest spoils quickly without cooling.', 'Sweet slices often top cakes and desserts.']}, 'carrot': {'group': 'vegetables', 'facts': ['A tapered orange root vegetable grows underground.', 'Feathery green foliage emerges above the soil.', 'Crunchy roots are commonly eaten raw.', 'Beta carotene contributes to the bright pigment.', 'Thin slices soften in soups and stews.']}, 'potato': {'group': 'vegetables', 'facts': ['A starchy edible tuber grows beneath the soil.', 'Sprouting eyes can produce new plants.', 'Boiling and baking soften dense white flesh.', 'Harvested tubers store energy for the plant.', 'Greened skin should not be eaten.']}, 'tomato': {'group': 'vegetables', 'facts': ['A juicy red garden fruit is used as a vegetable.', 'Numerous seeds sit in gel-filled chambers.', 'Vines are often supported with stakes.', 'Heating produces sauces with savory depth.', 'Green specimens gradually change color when ripe.']}, 'cucumber': {'group': 'vegetables', 'facts': ['A long green vegetable has watery crisp flesh.', 'Climbing vines bear yellow flowers.', 'Tiny seeds run along the soft center.', 'Thin slices cool salads and sandwiches.', 'Pickling preserves the harvest in brine.']}, 'onion': {'group': 'vegetables', 'facts': ['A layered bulb has a sharp aroma when cut.', 'Sulfur compounds make the eyes water.', 'Papery skin protects fleshy concentric layers.', 'Slow cooking creates a sweet brown flavor.', 'Bulbs can be stored in a dry pantry.']}, 'garlic': {'group': 'vegetables', 'facts': ['A pungent bulb divides into separate cloves.', 'Crushed cloves release strong sulfur aromas.', 'Individual segments are planted to grow new bulbs.', 'Roasting produces a mellow spreadable texture.', 'The plant belongs to the allium family.']}, 'broccoli': {'group': 'vegetables', 'facts': ['A green vegetable has clusters of unopened buds.', 'Thick stalks support branched florets.', 'Steaming softens the crisp stems.', 'Yellow flowers emerge if heads are left unharvested.', 'Leaves also contain edible nutrients.']}, 'cabbage': {'group': 'vegetables', 'facts': ['A leafy vegetable forms a tight round head.', 'Broad leaves overlap in dense layers.', 'Shredding produces the base of a slaw.', 'Fermentation creates a tangy preserved food.', 'Cool weather favors head development.']}, 'eggplant': {'group': 'vegetables', 'facts': ['A glossy purple vegetable has spongy flesh.', 'Tiny seeds remain embedded in the interior.', 'The skin becomes tender after roasting.', 'Warm soils support the flowering plant.', 'Slices absorb cooking oil readily.']}, 'pumpkin': {'group': 'vegetables', 'facts': ['A large orange squash has a hard outer rind.', 'A hollow center contains flat seeds.', 'Thick flesh is cooked into soups and pies.', 'Vines spread across garden ground.', 'Harvested fruit often decorates autumn displays.']}, 'bicycle': {'group': 'vehicles', 'facts': ['A two-wheeled vehicle moves by pedaling.', 'A chain commonly transfers power to a rear wheel.', 'Hand brakes slow the rotating rims or discs.', 'Balance and steering enable turning.', 'Riders often wear helmets for protection.']}, 'motorcycle': {'group': 'vehicles', 'facts': ['A powered two-wheeled road vehicle carries a rider.', 'A twist throttle controls engine power.', 'Leaning helps negotiate curves at speed.', 'Protective jackets reduce injuries in a fall.', 'Exhaust sound accompanies many fuel-driven models.']}, 'car': {'group': 'vehicles', 'facts': ['A four-wheeled road vehicle transports passengers.', 'Doors provide access to an enclosed cabin.', 'Steering wheels direct the front axle.', 'Seat belts restrain occupants during collisions.', 'A trunk holds luggage for journeys.']}, 'bus': {'group': 'vehicles', 'facts': ['A large road vehicle carries many seated passengers.', 'Fixed routes stop at designated locations.', 'Wide doors speed boarding and exit.', 'Public transit reduces the number of separate cars.', 'Standing riders often hold overhead rails.']}, 'train': {'group': 'vehicles', 'facts': ['Rail vehicles travel along fixed steel tracks.', 'Multiple carriages couple behind motive power.', 'Stations organize boarding along platforms.', 'Long routes transport people and freight.', 'Signals coordinate safe passage on shared lines.']}, 'tram': {'group': 'vehicles', 'facts': ['A street-running rail vehicle serves urban neighborhoods.', 'Tracks often share corridors with road traffic.', 'Overhead wires supply power to many systems.', 'Frequent stops enable short journeys.', 'The cabin typically has wide entry doors.']}, 'airplane': {'group': 'vehicles', 'facts': ['A fixed-wing aircraft flies using aerodynamic lift.', 'Engines provide forward thrust.', 'Airport runways support takeoff and landing.', 'Pressurized cabins carry travelers at altitude.', 'Flaps change wing performance at low speeds.']}, 'helicopter': {'group': 'vehicles', 'facts': ['A rotorcraft can hover over a single location.', 'Spinning overhead blades generate lift.', 'Vertical takeoff requires little runway space.', 'A tail rotor often balances turning forces.', 'Rescue teams use it to reach remote areas.']}, 'boat': {'group': 'vehicles', 'facts': ['A small watercraft floats on rivers or lakes.', 'Hull shape displaces water to provide buoyancy.', 'Paddles or engines produce forward motion.', 'A rudder or oar helps steer.', 'Passengers may wear personal flotation devices.']}, 'submarine': {'group': 'vehicles', 'facts': ['An underwater vessel can travel below the surface.', 'Ballast tanks change buoyancy for diving.', 'Thick hulls withstand external water pressure.', 'Sonar helps navigation without direct visibility.', 'Crews may remain submerged for extended periods.']}, 'piano': {'group': 'musical_instruments', 'facts': ['A keyboard instrument makes notes by striking strings.', 'Pedals sustain or alter the sound.', 'Black and white keys span multiple octaves.', 'A wooden soundboard amplifies vibrations.', 'Two hands can play independent melodic lines.']}, 'violin': {'group': 'musical_instruments', 'facts': ['A small string instrument is played beneath the chin.', 'A bow draws sound from four strings.', 'Fine finger placement changes the pitch.', 'A hollow wooden body resonates acoustically.', 'Orchestras commonly group many performers together.']}, 'guitar': {'group': 'musical_instruments', 'facts': ['A fretted string instrument is held against the torso.', 'Plucking or strumming sets strings vibrating.', 'Chords use several notes at once.', 'An acoustic wooden body amplifies vibrations.', 'Players often press strings along a fingerboard.']}, 'flute': {'group': 'musical_instruments', 'facts': ['A wind instrument makes sound across an air opening.', 'Fingered keys change the resonant air column.', 'It is often held horizontally while playing.', 'Bright high tones can carry over an orchestra.', 'Breath control shapes the musical phrase.']}, 'clarinet': {'group': 'musical_instruments', 'facts': ['A woodwind instrument uses a single vibrating reed.', 'The mouthpiece supports a flexible cane strip.', 'A cylindrical bore gives its characteristic tone.', 'Keys cover holes along a long tube.', 'The instrument has a broad low-to-high range.']}, 'saxophone': {'group': 'musical_instruments', 'facts': ['A metal woodwind uses a single-reed mouthpiece.', 'A curved bell projects its expressive sound.', 'Key pads open and close tone holes.', 'Jazz solos often feature this instrument.', 'Several sizes cover different registers.']}, 'trumpet': {'group': 'musical_instruments', 'facts': ['A brass instrument uses buzzing lips to create tones.', 'Three valves redirect air through tubing.', 'A flared bell projects a brilliant sound.', 'Players alter pitch through embouchure.', 'Fanfare melodies frequently use this instrument.']}, 'trombone': {'group': 'musical_instruments', 'facts': ['A brass instrument changes pitch using a long slide.', 'A large bell projects deep resonant tones.', 'Moving parallel tubes alters air-column length.', 'It often supports low orchestral harmonies.', 'Players coordinate arm movement with lip tension.']}, 'drum': {'group': 'musical_instruments', 'facts': ['A percussion instrument sounds when a membrane is struck.', 'Sticks or hands create rhythmic patterns.', 'Tension affects the pitch and timbre.', 'Different sizes produce varied low and high sounds.', "Repeated beats establish a song's pulse."]}, 'harp': {'group': 'musical_instruments', 'facts': ['A large frame instrument has many plucked strings.', 'The strings descend across an open triangular frame.', 'Pedals on concert versions change pitches.', 'Sweeping arpeggios create rippling effects.', 'Players use fingertips of both hands.']}, 'hammer': {'group': 'tools', 'facts': ['A striking tool has a heavy head and handle.', 'Repeated blows drive nails into wood.', 'A claw may pull embedded fasteners.', 'The grip transfers force from an arm swing.', 'Metal heads can damage delicate surfaces.']}, 'screwdriver': {'group': 'tools', 'facts': ['A hand tool rotates threaded fasteners.', 'Shaped tips fit matching slots in screw heads.', 'Torque transfers through a narrow shaft.', 'Interchangeable bits handle different patterns.', 'The handle is turned rather than struck.']}, 'wrench': {'group': 'tools', 'facts': ['A gripping tool turns nuts and bolts.', 'A shaped jaw engages flat-sided hardware.', 'Adjustable designs fit several sizes.', 'Leverage helps loosen stubborn connections.', 'Mechanics frequently use it around engines.']}, 'pliers': {'group': 'tools', 'facts': ['A hinged hand tool grips objects between jaws.', 'Squeezing two handles increases clamping force.', 'Some designs cut thin wires.', 'Serrated metal faces prevent slipping.', 'The pivot allows strong precise holds.']}, 'saw': {'group': 'tools', 'facts': ['A cutting tool has a line of sharp teeth.', 'Back-and-forth strokes remove material.', 'Different blades cut wood or metal.', 'The kerf leaves a narrow channel.', 'Guides can help keep the cut straight.']}, 'chisel': {'group': 'tools', 'facts': ['A narrow bladed tool carves or shapes solid material.', 'The cutting edge is guided into the surface.', 'A mallet may tap the handle.', 'Woodworkers use it for joints and grooves.', 'Sharpening preserves a clean controlled cut.']}, 'scissors': {'group': 'tools', 'facts': ['A hand tool cuts with two crossing blades.', 'Finger loops control the hinged handles.', 'Shearing action divides paper or fabric.', 'The pivot holds sharpened edges together.', 'Right-handed and left-handed models differ.']}, 'knife': {'group': 'tools', 'facts': ['A handheld cutting tool has one sharp blade.', 'Edge geometry determines slicing performance.', 'A handle keeps fingers behind the cutting edge.', 'Kitchen versions prepare food ingredients.', 'A protective sheath can cover the point.']}, 'drill': {'group': 'tools', 'facts': ['A powered or manual tool creates round holes.', 'A rotating bit removes material.', 'A chuck secures interchangeable attachments.', 'Speed settings suit different surfaces.', 'Some designs also drive threaded fasteners.']}, 'shovel': {'group': 'tools', 'facts': ['A long-handled tool moves soil or sand.', 'A broad blade enters loose ground.', 'Foot pressure can push the edge deeper.', 'Gardeners use it to dig planting holes.', 'Levering the handle lifts a loaded scoop.']}, 'refrigerator': {'group': 'home_appliances', 'facts': ['An insulated cabinet keeps food cold.', 'A compressor moves heat out of the interior.', 'Shelves organize perishables in compartments.', 'Door seals reduce the entry of warm air.', 'Thermostats regulate storage temperature.']}, 'oven': {'group': 'home_appliances', 'facts': ['An enclosed appliance cooks food with sustained heat.', 'Metal racks support trays and dishes.', 'Baking transforms dough into crusty bread.', 'Insulated walls limit heat loss.', 'Temperature dials control roasting conditions.']}, 'microwave': {'group': 'home_appliances', 'facts': ['A countertop appliance heats food using electromagnetic waves.', 'Rotating trays help distribute energy.', 'Short cycles rewarm leftovers rapidly.', 'A shielded door prevents radiation leakage.', 'Moist food warms by dielectric heating.']}, 'toaster': {'group': 'home_appliances', 'facts': ['A small kitchen appliance browns sliced bread.', 'Electric elements heat both sides.', 'A spring often raises the food when finished.', 'Browning settings change heating duration.', 'Crumbs collect in a removable tray.']}, 'blender': {'group': 'home_appliances', 'facts': ['A kitchen appliance mixes food with rapidly spinning blades.', 'A lidded jar contains liquids during operation.', 'Fruit and ice can become a smooth drink.', 'Different speeds change texture.', 'The motor drives a cutting assembly at the bottom.']}, 'vacuum': {'group': 'home_appliances', 'facts': ['A cleaning appliance removes dust with suction.', 'An electric motor moves air through a chamber.', 'Filters trap fine particles before exhaust.', 'Hose attachments reach narrow spaces.', 'A collection bag or bin holds debris.']}, 'washer': {'group': 'home_appliances', 'facts': ['A laundry appliance cleans clothing with water.', 'A rotating drum agitates fabric.', 'Detergent loosens soil and oils.', 'Spin cycles remove much of the remaining moisture.', 'Program settings select wash temperature and duration.']}, 'dryer': {'group': 'home_appliances', 'facts': ['A laundry appliance removes water from washed clothes.', 'Heated airflow passes through tumbling fabric.', 'A lint filter catches loose fibers.', 'Moisture sensors may stop the cycle automatically.', 'Venting or condensation manages damp exhaust.']}, 'dishwasher': {'group': 'home_appliances', 'facts': ['A kitchen appliance cleans plates with sprayed water.', 'Interior racks hold utensils and dishes.', 'Detergent dissolves food residue during cycles.', 'Hot rinses help with drying and sanitation.', 'A pump circulates water through spray arms.']}, 'fan': {'group': 'home_appliances', 'facts': ['An electric appliance circulates air with rotating blades.', 'Adjustable speeds change airflow strength.', 'A grille protects users from moving parts.', 'Oscillation spreads the breeze across a room.', 'Ventilation can improve comfort without cooling air itself.']}}

In [ ]:
WIKI_TITLES={
 'apple':'Apple','orange':'Orange (fruit)','lime':'Lime (fruit)','lemon':'Lemon','peach':'Peach',
 'pear':'Pear','grape':'Grape','strawberry':'Strawberry','pineapple':'Pineapple','banana':'Banana',
 'seal':'Pinniped','whale':'Whale','shark':'Shark','crab':'Crab','lobster':'Lobster',
 'fan':'Fan (machine)','washer':'Washing machine','dryer':'Clothes dryer',
 'vacuum':'Vacuum cleaner','oven':'Oven','microwave':'Microwave oven','blender':'Blender',
 'car':'Car','bus':'Bus','train':'Train','tram':'Tram','boat':'Boat',
 'drill':'Drill','saw':'Saw (tool)','chisel':'Chisel','hammer':'Hammer',
 'ant':'Ant','cricket':'Cricket (insect)','moth':'Moth','spider':'Spider','scorpion':'Scorpion',
 'rose':'Rose','violin':'Violin','piano':'Piano','guitar':'Guitar'
}

# Wikis provide article revisions for provenance. Article content may be revised between runs:
# the collected frozen stimulus_design.csv is committed before extraction.
def retrieve_article(domain,title):
    url=f'https://{domain}.wikipedia.org/w/api.php'
    params={'action':'query','prop':'extracts|revisions','explaintext':1,'redirects':1,
            'rvprop':'ids|timestamp','titles':title,'format':'json','formatversion':2}
    for attempt in range(4):
        try:
            r=SOURCE_SESSION.get(url,params=params,timeout=60)
            if r.status_code in (429,500,502,503,504):
                time.sleep(2**attempt);continue
            r.raise_for_status()
            p=r.json()['query']['pages'][0]
            if p.get('missing') or p.get('invalid'):return None
            extract=p.get('extract','')
            if not extract or 'may refer to:' in extract[:350].lower():return None
            return {'text':extract,'title':p['title'],'rev':p.get('revisions',[{}])[0].get('revid'),
                    'url':f'https://{domain}.wikipedia.org/wiki/'+requests.utils.quote(p['title'].replace(' ','_'))}
        except Exception:
            if attempt==3:return None
            time.sleep(2**attempt)
    return None

def candidate_sentences(text,name,title):
    # No fabricated or templated facts. Reject headings, very short lines, name mentions,
    # citation artifacts and probable lists. This screen is conservative, not perfect.
    text=re.sub(r'\[[0-9]+\]','',text)
    text=re.sub(r'(?m)^=+.*?=+\s*$',' ',text)
    text=re.sub(r'\n{2,}',' ',text)
    chunks=re.split(r'(?<=[.!?])\s+(?=[A-Z0-9\(])',text)
    output=[];seen=set()
    terms=[name,title,title.split('(')[0].strip()]
    terms=[re.escape(t) for t in terms if len(t)>=3]
    forbidden=re.compile(r'(?i)\b(?:'+'|'.join(terms)+r')s?\b')
    for sentence in chunks:
        sentence=unicodedata.normalize('NFKC',re.sub(r'\s+',' ',sentence.strip()))
        if len(sentence)<52 or len(sentence)>280:continue
        if len(sentence.split())<9 or len(sentence.split())>43:continue
        if forbidden.search(sentence):continue
        if any(x in sentence.lower() for x in ['see also','external links','references','bibliography','isbn','doi:','http']):continue
        if sentence.count(';')>2:continue
        normalized=re.sub(r'[^a-z0-9]','',sentence.casefold())
        if normalized in seen:continue
        seen.add(normalized)
        output.append(sentence)
    return output

raw_rows=[];coverage=[]
def tokset(sentence):return set(re.findall(r'[a-z]+',sentence.lower()))
def too_similar(sentence,others,threshold=.55):
    a=tokset(sentence)
    return any(len(a & tokset(b))/max(1,len(a | tokset(b)))>=threshold for b in others)

for name,item in tqdm(BANK.items(),desc='Collect diverse factual sentences'):
    title=WIKI_TITLES.get(name,name.capitalize())
    train_article=retrieve_article('en',title)
    test_article=retrieve_article('simple',title)
    if test_article is None and title!=name.capitalize():
        test_article=retrieve_article('simple',name.capitalize())
    en_all=candidate_sentences(train_article['text'],name,train_article['title']) if train_article else []
    simple_all=candidate_sentences(test_article['text'],name,test_article['title']) if test_article else []
    train_sentences=en_all[:TRAIN_PER_CONCEPT]
    eval_items=[]
    # Prefer truly source-held-out evaluation sentences from Simple English Wikipedia.
    for sentence in simple_all:
        if too_similar(sentence,train_sentences) or too_similar(sentence,[x[0] for x in eval_items]):continue
        eval_items.append((sentence,test_article,'simple_wikipedia'))
        if len(eval_items)==EVAL_PER_CONCEPT:break
    n_simple=len(eval_items)
    # Fallback for coverage: later, different sentences from English Wikipedia.
    # Explicitly mark these as same-article holdout (weaker evidence).
    if len(eval_items)<EVAL_PER_CONCEPT:
        for sentence in en_all[TRAIN_PER_CONCEPT:]:
            if too_similar(sentence,train_sentences) or too_similar(sentence,[x[0] for x in eval_items]):continue
            eval_items.append((sentence,train_article,'en_wikipedia_fallback'))
            if len(eval_items)==EVAL_PER_CONCEPT:break
    eligible=len(train_sentences)==TRAIN_PER_CONCEPT and len(eval_items)==EVAL_PER_CONCEPT
    coverage.append({'concept':name,'group':item['group'],
                     'en_title':train_article['title'] if train_article else '',
                     'simple_title':test_article['title'] if test_article else '',
                     'train_count':len(train_sentences),'eval_count':len(eval_items),
                     'eval_from_simple_count':n_simple,'eval_from_en_fallback_count':len(eval_items)-n_simple,
                     'eligible':eligible,'en_rev':train_article['rev'] if train_article else None,
                     'simple_rev':test_article['rev'] if test_article else None})
    if not eligible:continue
    for family,fact in enumerate(train_sentences):
        raw_rows.append({'concept':name,'group':item['group'],'split':'train','family':family,
                         'fact':fact,'source_url':train_article['url'],'source_revision':train_article['rev'],
                         'source_tier':'en_wikipedia_train'})
    for family,(fact,article,tier) in enumerate(eval_items):
        raw_rows.append({'concept':name,'group':item['group'],'split':'eval','family':family,
                         'fact':fact,'source_url':article['url'],'source_revision':article['rev'],
                         'source_tier':tier})

coverage_df=pd.DataFrame(coverage);coverage_df.to_csv(OUT/'data_coverage.csv',index=False)
facts_df=pd.DataFrame(raw_rows)
if len(facts_df):facts_df.to_csv(OUT/'sourced_facts.csv',index=False)
print('Eligible concepts:',int(coverage_df.eligible.sum()),'/100')
print('Coverage by group:')
print(coverage_df.groupby('group').eligible.agg(['sum','count']))
assert int(coverage_df.eligible.sum())>=20, 'Fewer than 20 concepts have 8+8 source sentences. Review coverage; do NOT weaken validation silently.'
# Text sharing license: Wikipedia CC BY-SA; retain per-fact provenance URLs and article revision.


## 2. Freeze independent descriptions; choose semantic-distance bands *before* Qwen scoring

Two different websites provide train/evaluation sentences; however both can describe the same facts and may share historical text. The overlap filter catches some, not all, near-duplicates. Auditing is mandatory before thesis claims. **Distance** is cosine distance on title + training text only, and the six Y comparators are frozen independently of hidden-state scores.


In [ ]:
eligible=sorted(coverage_df.loc[coverage_df.eligible,'concept'])
concepts=sorted(facts_df.concept.unique())
GROUPS=defaultdict(list)
for c in concepts:GROUPS[BANK[c]['group']].append(c)
embedder=SentenceTransformer(EMBEDDER,device='cpu')
train_text={c:' '.join(facts_df.loc[(facts_df.concept==c)&(facts_df.split=='train'),'fact'].tolist()[:3]) for c in concepts}
texts=[f'{c}. {train_text[c]}' for c in concepts]
emb=embedder.encode(texts,normalize_embeddings=True,show_progress_bar=True)
distance=1-np.asarray(emb)@np.asarray(emb).T
idx={c:i for i,c in enumerate(concepts)}
rows=[]
for x in concepts:
    members=[y for y in GROUPS[BANK[x]['group']] if y!=x]
    if len(members)<6:continue
    ordered=sorted(members,key=lambda y:(float(distance[idx[x],idx[y]]),y))
    # Disjoint near, medium and far picks, with 2 comparators per band.
    selections={'near':ordered[:2],'medium':ordered[(len(ordered)-2)//2:(len(ordered)-2)//2+2], 'far':ordered[-2:]}
    if len({v for group in selections.values() for v in group})!=6:continue
    for band,ys in selections.items():
        for y in ys:
            rows.append({'target':x,'comparator':y,'group':BANK[x]['group'],'band':band,'cosine_distance':float(distance[idx[x],idx[y]])})
comparisons=pd.DataFrame(rows)
assert len(comparisons)>0, 'Not enough within-category eligible concepts; inspect coverage report.'
comparisons.to_csv(OUT/'candidate_comparators_frozen.csv',index=False)
print('Directed comparators:',len(comparisons),'distinct targets:',comparisons.target.nunique())
print(comparisons.groupby('band').cosine_distance.describe())

records=[]
for r in facts_df.itertuples():
    # One prompt per genuinely distinct sentence (no duplicated wrappers).
    user_prompt='Read the factual note carefully.\n\nNote: '+r.fact+'\n\nBriefly summarize the note.'
    records.append({'concept':r.concept,'group':r.group,'split':r.split,'family':int(r.family),
                    'fact':r.fact,'source_url':r.source_url,'source_revision':r.source_revision,'user_prompt':user_prompt})
prompts_df=pd.DataFrame(records)
assert not prompts_df.user_prompt.duplicated().any(), 'Duplicate facts across target concepts; inspect and curate.'
prompts_df.to_csv(OUT/'stimulus_design.csv',index=False)
PROMPT_HASH=hashlib.sha256(prompts_df.to_csv(index=False).encode()).hexdigest()
CONFIG={'model':MODEL_ID,'revision':REVISION,'embedder':EMBEDDER,'prompt_hash':PROMPT_HASH,
        'layers':LAYERS,'position':'assistant_start','concept_count':len(concepts),
        'batch_size':BATCH_SIZE,'train_families':8,'eval_families':8,'run_id':RUN_ID}
print('Prompts:',len(prompts_df),'hash:',PROMPT_HASH)
# Freeze source provenance + generated design as commit, before GPU inference.
for file in ['data_coverage.csv','sourced_facts.csv','candidate_comparators_frozen.csv','stimulus_design.csv']:
    path=OUT/file
    github_save(f'{DEST}/{file}',path.read_bytes(),'Freeze richer Concept Probe source inputs: '+file)
github_save(f'{DEST}/config.json',json.dumps(CONFIG,indent=2).encode(),'Freeze input identity for richer Concept Probe run')


## 3. Extract Qwen hidden states with remote checkpoint recovery

Resuming an interrupted run restores existing GitHub checkpoint archives and verifies their prompt hash and matrix shape. Never delete or overwrite old checkpoints to resolve a conflict; use a new `RUN_ID` for a changed dataset.


In [ ]:
assert torch.cuda.is_available(),'Select GPU / T4 runtime in Colab.'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,torch_dtype=torch.float32,attn_implementation='eager').to('cuda').eval()
assert model.config.num_hidden_layers==24
HIDDEN=model.config.hidden_size

@torch.inference_mode()
def extract_one(prompt):
    t=tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
    h=model(input_ids=t,output_hidden_states=True,use_cache=False).hidden_states
    return np.stack([h[layer][0,-1].float().cpu().numpy() for layer in LAYERS])

N=len(prompts_df);N_BATCH=math.ceil(N/BATCH_SIZE);N_GROUP=math.ceil(N_BATCH/BATCHES_PER_CHECKPOINT)
def groups(g):return range(g*BATCHES_PER_CHECKPOINT,min((g+1)*BATCHES_PER_CHECKPOINT,N_BATCH))
def local_batch(b):return CACHE/f'batch_{b:03d}.npz'
def remote_chunk(g):return f'{DEST}/checkpoints/chunk_{g:03d}.zip'
def check_batch(b,data):
    with np.load(io.BytesIO(data),allow_pickle=False) as z:
        assert str(z['prompt_hash'].item())==PROMPT_HASH
        assert int(z['start'])==b*BATCH_SIZE and int(z['stop'])==min(N,(b+1)*BATCH_SIZE)
        feat=z['features']
        assert feat.shape==(min(N,(b+1)*BATCH_SIZE)-b*BATCH_SIZE,len(LAYERS),HIDDEN)
        assert np.isfinite(feat).all()

def recover_group(g):
    data=github_get(remote_chunk(g))
    if data is None:return False
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        assert json.loads(z.read('config.json'))==CONFIG, 'Run configuration changed. Change RUN_ID.'
        names={f'batch_{b:03d}.npz' for b in groups(g)}|{'config.json'}
        assert set(z.namelist())==names
        for b in groups(g):
            name=f'batch_{b:03d}.npz';blob=z.read(name)
            check_batch(b,blob)
            local_batch(b).write_bytes(blob)
    print('Recovered checkpoint group',g)
    return True

def publish_group(g):
    if not all(local_batch(b).exists() for b in groups(g)):return False
    if recover_group(g):return True
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('config.json',json.dumps(CONFIG,sort_keys=True))
        for b in groups(g):
            blob=local_batch(b).read_bytes();check_batch(b,blob)
            z.writestr(f'batch_{b:03d}.npz',blob)
    github_save(remote_chunk(g),buf.getvalue(),f'Checkpoint richer Concept Probe group {g:03d}')
    return True

for g in range(N_GROUP):recover_group(g)
for b in tqdm(range(N_BATCH),desc='Qwen hidden states'):
    path=local_batch(b)
    if path.exists():check_batch(b,path.read_bytes());continue
    start=b*BATCH_SIZE;stop=min(N,start+BATCH_SIZE)
    features=np.stack([extract_one(p) for p in prompts_df.user_prompt.iloc[start:stop]])
    buf=io.BytesIO();np.savez_compressed(buf,prompt_hash=np.array(PROMPT_HASH),start=np.array(start),stop=np.array(stop),features=features)
    path.write_bytes(buf.getvalue())
    if (b+1)%BATCHES_PER_CHECKPOINT==0 or b+1==N_BATCH:publish_group(b//BATCHES_PER_CHECKPOINT)
for g in range(N_GROUP):assert publish_group(g)
FEATURES=np.concatenate([np.load(local_batch(b),allow_pickle=False)['features'] for b in range(N_BATCH)])
assert FEATURES.shape==(N,len(LAYERS),HIDDEN)
print('Feature tensor:',FEATURES.shape)


## 4. Train probes using 8 truly different training statements, score on 8 independent-source evaluation statements

We evaluate all selected unique unordered contrasts at layer 22/23/24. Primary layer 24 gets 8 leave-one-training-family-out stability fits. Bootstrap uncertainty is over evaluation fact families; there is no wrapper-based pseudo replication. Label-shuffle is a control (not a confirmatory statistical test after selection).


In [ ]:
INDEX={}
for i,r in enumerate(prompts_df.itertuples()):INDEX[(r.concept,r.split,int(r.family))]=i
assert len(INDEX)==len(prompts_df)

def vectors(concept,split,li):
    return np.stack([FEATURES[INDEX[(concept,split,f)],li] for f in range(8)])

def fit_probe(a,b):
    X=np.concatenate([a,b]);y=np.array([1]*len(a)+[0]*len(b))
    scaler=StandardScaler().fit(X)
    classifier=LogisticRegression(C=0.1,solver='liblinear',max_iter=1000,random_state=SEED).fit(scaler.transform(X),y)
    return scaler,classifier

def score_probe(scaler,fit,a,b):
    X=np.concatenate([a,b]);y=np.array([1]*len(a)+[0]*len(b))
    return float(roc_auc_score(y,fit.decision_function(scaler.transform(X))))

unique_pairs=sorted(set(tuple(sorted((r.target,r.comparator))) for r in comparisons.itertuples()))
scan=[]
for a,b in tqdm(unique_pairs,desc='Probe contrast screening'):
    row={'pair':'__'.join([a,b]),'a':a,'b':b,'group':BANK[a]['group']}
    for li,layer in enumerate(LAYERS):
        ta,tb=vectors(a,'train',li),vectors(b,'train',li)
        ea,eb=vectors(a,'eval',li),vectors(b,'eval',li)
        scaler,clf=fit_probe(ta,tb)
        row[f'L{layer}_auc']=score_probe(scaler,clf,ea,eb)
        if layer==24:
            loo=[]
            for i in range(8):
                keep=[j for j in range(8) if j!=i]
                s,c=fit_probe(ta[keep],tb[keep]);loo.append(score_probe(s,c,ea,eb))
            row['L24_loo_min']=float(min(loo));row['L24_loo_mean']=float(np.mean(loo))
    scan.append(row)
scan_df=pd.DataFrame(scan)
scan_df['screen_pass']=(scan_df.L24_auc>=.85)&(scan_df.L24_loo_min>=.80)&(scan_df.L23_auc>=.75)
scan_df.to_csv(OUT/'pair_screening.csv',index=False)
print('Screened:',len(scan_df),'pass:',int(scan_df.screen_pass.sum()))


In [ ]:
# Label shuffle with 100 fixed permutations for all screen-pass pairs.
# 16 training facts (8 X + 8 Y). Each shuffled fit preserves class counts.
null_rows=[]
for r in tqdm(scan_df.loc[scan_df.screen_pass].itertuples(),total=int(scan_df.screen_pass.sum()),desc='Shuffle controls'):
    a,b=r.a,r.b
    ta,tb=vectors(a,'train',2),vectors(b,'train',2)
    ea,eb=vectors(a,'eval',2),vectors(b,'eval',2)
    X=np.concatenate([ta,tb]);scaler=StandardScaler().fit(X)
    Xtrain=scaler.transform(X);Xeval=scaler.transform(np.concatenate([ea,eb]))
    yeval=np.array([1]*8+[0]*8)
    rng=np.random.default_rng(int(hashlib.sha256((r.pair+str(SEED)).encode()).hexdigest()[:8],16))
    shuffled_auc=[]
    for i in range(100):
        labels=np.array([1]*8+[0]*8);rng.shuffle(labels)
        c=LogisticRegression(C=0.1,solver='liblinear',max_iter=1000,random_state=SEED).fit(Xtrain,labels)
        shuffled_auc.append(float(roc_auc_score(yeval,c.decision_function(Xeval))))
    null_rows.append({'pair':r.pair,'shuffle_n':len(shuffled_auc),'shuffle_tail_p_plus_one':float((1+sum(x>=r.L24_auc-1e-12 for x in shuffled_auc))/(len(shuffled_auc)+1)),
                      'shuffle_auc_mean':float(np.mean(shuffled_auc)),'observed_auc':float(r.L24_auc)})
null_df=pd.DataFrame(null_rows,columns=['pair','shuffle_n','shuffle_tail_p_plus_one','shuffle_auc_mean','observed_auc'])
scan_df=scan_df.merge(null_df,on='pair',how='left')
scan_df['exploratory_candidate']=scan_df.screen_pass&(scan_df.shuffle_tail_p_plus_one<=.05)
scan_df.to_csv(OUT/'pair_screening_with_controls.csv',index=False)
print('Exploratory pairs after shuffle:',int(scan_df.exploratory_candidate.sum()))


## 5. Count viable target concepts by distinct Y and independent distance bands

This screen counts **distinct target concepts**, not merely pairs. Avoid interpreting a 20-concept shortlist as 20 statistically confirmed measurements: selected pairs must be tested again on an *additional* set of fresh, independently reviewed texts.


In [ ]:
keyed=scan_df.set_index('pair')
paired=comparisons.copy()
paired['pair']=paired.apply(lambda r:'__'.join(sorted((r.target,r.comparator))),axis=1)
paired=paired.merge(scan_df[['pair','L22_auc','L23_auc','L24_auc','L24_loo_min','screen_pass','exploratory_candidate','shuffle_tail_p_plus_one']],on='pair',how='left')
paired.to_csv(OUT/'distance_by_target_comparator.csv',index=False)
summary=paired.groupby('band').agg(n=('L24_auc','size'),mean_auc=('L24_auc','mean'),median_auc=('L24_auc','median'),mean_distance=('cosine_distance','mean'),stable=('exploratory_candidate','sum')).reset_index()
summary.to_csv(OUT/'distance_band_summary.csv',index=False)
rows=[]
for x,g in paired.groupby('target'):
    good=g[g.exploratory_candidate.fillna(False)]
    rows.append({'target':x,'group':BANK[x]['group'],'comparisons_tested':len(g),
                 'stable_comparators':good.comparator.nunique(),'stable_bands':good.band.nunique(),
                 'comparators':', '.join(good.comparator.tolist()),
                 'near_mean_auc':float(g.loc[g.band=='near','L24_auc'].mean()),
                 'medium_mean_auc':float(g.loc[g.band=='medium','L24_auc'].mean()),
                 'far_mean_auc':float(g.loc[g.band=='far','L24_auc'].mean()),
                 'two_bands_stable':good.comparator.nunique()>=2 and good.band.nunique()>=2})
target_df=pd.DataFrame(rows).sort_values(['two_bands_stable','stable_comparators','target'],ascending=[False,False,True])
target_df.to_csv(OUT/'target_candidates.csv',index=False)
print('Distinct target concepts stable in ≥2 distance bands:',int(target_df.two_bands_stable.sum()), '/',len(target_df))
print(summary.to_string(index=False))
print(target_df.head(25).to_string(index=False))
fig,ax=plt.subplots(figsize=(8,5))
vals=[paired.loc[paired.band==band,'L24_auc'].dropna().to_numpy() for band in ['near','medium','far']]
ax.boxplot(vals,tick_labels=['near','medium','far']);ax.set_ylabel('ROC-AUC');ax.set_ylim(0,1.02)
ax.set_title('Relative concept decodability vs semantic distance')
fig.tight_layout();fig.savefig(OUT/'distance_auc_boxplot.png',dpi=180);plt.show()


In [ ]:
manifest={**CONFIG,'status':'COMPLETED_EXPLORATORY_SCREEN','created_utc':datetime.now(timezone.utc).isoformat(),
          'gpu':torch.cuda.get_device_name(0),'prompts':N,'concepts_eligible':len(concepts),'compared_targets':comparisons.target.nunique(),
          'directed_comparisons':len(comparisons),'unique_unordered_pairs':len(scan_df),'screen_pass_pairs':int(scan_df.screen_pass.sum()),
          'shuffle_tested_pairs':len(null_df),'exploratory_pair_pass':int(scan_df.exploratory_candidate.sum()),
          'two_distance_bands_target_count':int(target_df.two_bands_stable.sum()),
          'caveats':['Wikipedia sources are distinct websites, not independent experimental random samples.',
                     'Evaluation claims can be topically overlapping and are automatically filtered only; manually audit selected targets.',
                     'Source-holdout and distance bands do not establish causal effect of semantic distance.',
                     'AUC and shortlist selection use the same evaluation split; confirm with genuinely new descriptions.',
                     'Probe reads X-versus-Y relative decodability, not absolute activation or suppression.',
                     'No multiple-testing correction has been applied.']}
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
report=f"""# Qwen Concept Probe: richer 100-concept study

- Eligible concepts (8 train + 8 evaluation sentences): **{len(concepts)} / 100**
- Sources: English Wikipedia (train); Simple English Wikipedia preferred for evaluation, with explicitly tagged same-article fallback. See per-sentence provenance in sourced_facts.csv.
- Model: {MODEL_ID} @ {REVISION}
- Unique prompts: {N}
- Unique unordered pairs: **{len(scan_df)}**
- Screen-passing pairs: **{int(scan_df.screen_pass.sum())}**
- Screen + shuffle-passing pairs: **{int(scan_df.exploratory_candidate.sum())}**
- Distinct targets passing across 2 distance bands: **{int(target_df.two_bands_stable.sum())}** (target ~20)

## Distance-banded performance

{summary.to_markdown(index=False)}

## Best targets

{target_df.head(30).to_markdown(index=False)}

**Interpretation:** Exploratory *pair-relative decodability*, not absolute concept activation. Manually review texts and run a preregistered additional held-out confirmation before thesis claims.
"""
(OUT/'analysis_summary.md').write_text(report)
archive=OUT/'qwen_100concept_richer_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=archive:z.write(p,p.name)
for filename in ['manifest.json','analysis_summary.md','target_candidates.csv','pair_screening_with_controls.csv','distance_band_summary.csv','data_coverage.csv']:
    github_save(f'{DEST}/{filename}',(OUT/filename).read_bytes(),'Save richer Concept Probe result '+filename)
github_save(f'{DEST}/{archive.name}',archive.read_bytes(),'Save complete richer Concept Probe result archive')
print('Results saved: https://github.com/'+REPO+'/tree/main/'+DEST)
